# 12 Hypothesis Testing

In Chapter 11, we used confidence intervals to quantify uncertainty around an estimate.

Now we ask a different question:

> **Is the observed sample evidence strong enough to challenge a claim about the population?**

This is the central idea of **hypothesis testing**.

This chapter introduces the general framework. Chapter 13 will then focus specifically on the t-test.

## Learning Objectives

After completing this chapter, you will be able to:

- define a null hypothesis and an alternative hypothesis;
- understand one-sided and two-sided hypotheses;
- understand significance level $\alpha$;
- understand the logic of a test statistic;
- interpret a p-value;
- make a reject / fail-to-reject decision;
- distinguish statistical significance from practical importance;
- understand Type I and Type II errors;
- understand the relationship between confidence intervals and hypothesis tests;
- avoid common hypothesis-testing interpretation mistakes.

## 1. Statistical Question

Imagine a fictional customer-support organization.

Its historical target is:

$$
\mu_0 = 30\text{ minutes}
$$

for mean response time.

We collect a sample of recent support cases.

The question is:

> **Does the sample provide evidence that the population mean response time is different from 30 minutes?**

This is not the same as simply asking whether the sample mean is exactly 30.

Random samples naturally vary.

Hypothesis testing asks whether the observed difference is large enough, relative to sampling uncertainty, to be considered unusual under a specified claim.

## 2. Null and Alternative Hypotheses

The **null hypothesis** represents the claim we evaluate as the reference model:

$$
H_0:\mu=30
$$

The **alternative hypothesis** represents the competing claim:

$$
H_1:\mu\neq30
$$

This is a **two-sided test** because departures in either direction matter.

The hypotheses concern the **population parameter $\mu$**, not merely the observed sample mean $\bar{x}$.

## 3. Dataset

`12_hypothesis_testing.csv` contains 120 synthetic customer-support cases.

Columns:

- `case_id`
- `channel`
- `response_time_minutes`

The observations are fictional and generated solely for education.

The dataset was intentionally generated with response times centered somewhat above 30 minutes so that we have a useful hypothesis-testing example. It is **not real operational data**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm, t

cwd = Path.cwd().resolve()
root = next(
    (p for p in [cwd, *cwd.parents] if (p / "data").is_dir()),
    None,
)

if root is None:
    raise FileNotFoundError("Could not find project root containing data/.")

df = pd.read_csv(root / "data" / "12_hypothesis_testing.csv")
df.head()

In [ ]:
values = df["response_time_minutes"]

n = len(values)
x_bar = values.mean()
s = values.std(ddof=1)

print(f"Sample size (n): {n}")
print(f"Sample mean (x̄): {x_bar:.2f}")
print(f"Sample SD (s): {s:.2f}")

## 4. Sampling Variation Matters

Even if:

$$
H_0:\mu=30
$$

is true, a random sample will almost never have:

$$
\bar{x}=30
$$

exactly.

So the question is not:

> Is the sample mean different from 30?

Instead:

> **How unusual would this sample result be if $H_0$ were true?**

That question is the heart of hypothesis testing.

## 5. Test Statistic

A test statistic measures the observed difference relative to its expected sampling variability.

For a one-sample mean when the population standard deviation is unknown:

$$
t=
\frac{\bar{x}-\mu_0}
{s/\sqrt{n}}
$$

The denominator is the estimated standard error.

A large absolute value of $t$ means the observed sample mean is far from the null value relative to sampling uncertainty.

In [ ]:
mu_0 = 30

standard_error = s / np.sqrt(n)

t_statistic = (x_bar - mu_0) / standard_error

print(f"Standard error: {standard_error:.3f}")
print(f"t statistic:    {t_statistic:.3f}")

## 6. The Null Distribution

If $H_0$ is true and the assumptions of the one-sample t procedure are appropriate, the test statistic follows a t-distribution with:

$$
df=n-1
$$

The null distribution tells us which test-statistic values are common and which are unusual under $H_0$.

In [ ]:
dfree = n - 1

x = np.linspace(-4, 4, 500)
density = t.pdf(x, df=dfree)

plt.figure(figsize=(8, 5))
plt.plot(x, density)
plt.axvline(t_statistic, linestyle="--", label=f"Observed t = {t_statistic:.2f}")
plt.axvline(-abs(t_statistic), linestyle="--")
plt.title("t Distribution Under the Null Hypothesis")
plt.xlabel("t")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 7. What Is a p-value?

For this two-sided test, the p-value asks:

> **Assuming $H_0$ is true, how likely is a test statistic at least as extreme as the one we observed, in either direction?**

For a two-sided t-test:

$$
p
=
2P(T\geq|t_{\text{obs}}|)
$$

The p-value is calculated **under the assumption that the null hypothesis is true**.

In [ ]:
p_value = 2 * t.sf(
    abs(t_statistic),
    df=dfree,
)

print(f"Two-sided p-value: {p_value:.6f}")

## 8. Significance Level

Before evaluating the result, we choose a **significance level**:

$$
\alpha
$$

A common value is:

$$
\alpha=0.05
$$

Decision rule:

- if $p<\alpha$: **reject $H_0$**;
- if $p\geq\alpha$: **fail to reject $H_0$**.

Notice the wording:

> **Fail to reject** does not mean **prove that $H_0$ is true**.

In [ ]:
alpha = 0.05

if p_value < alpha:
    decision = "Reject H0"
else:
    decision = "Fail to reject H0"

print(f"α = {alpha}")
print(f"p = {p_value:.6f}")
print(f"Decision: {decision}")

## 9. Interpreting the Result

If $p<0.05$, the sample result would be relatively unusual under the null model.

We therefore reject:

$$
H_0:\mu=30
$$

at the 5% significance level.

A careful conclusion is:

> **The sample provides statistically significant evidence that the population mean response time differs from 30 minutes.**

This conclusion does **not** tell us automatically whether the difference is operationally important.

## 10. What a p-value Is NOT

A p-value is frequently misunderstood.

It is **not**:

$$
P(H_0\text{ is true})
$$

It is also not the probability that the result occurred “by chance.”

Instead, it describes how extreme the observed evidence is **under the null hypothesis and the assumed statistical model**.

A small p-value means the observed data are relatively incompatible with the null model.

It does not directly give the probability that a hypothesis is true or false.

## 11. Two-Sided vs One-Sided Tests

### Two-sided

$$
H_0:\mu=30
$$

$$
H_1:\mu\neq30
$$

Use this when differences in either direction matter.

### Right-sided

$$
H_1:\mu>30
$$

Use this when the research question specifically concerns an increase.

### Left-sided

$$
H_1:\mu<30
$$

Use this when the research question specifically concerns a decrease.

The direction should be determined by the research question **before examining the result**, not chosen afterward merely to obtain a smaller p-value.

## 12. Compare One-Sided and Two-Sided p-values

In [ ]:
two_sided_p = 2 * t.sf(abs(t_statistic), df=dfree)
right_sided_p = t.sf(t_statistic, df=dfree)
left_sided_p = t.cdf(t_statistic, df=dfree)

print(f"Two-sided p-value:  {two_sided_p:.6f}")
print(f"Right-sided p-value:{right_sided_p:.6f}")
print(f"Left-sided p-value: {left_sided_p:.6f}")

## 13. Type I Error

A **Type I error** occurs when we reject $H_0$ even though $H_0$ is actually true.

$$
\text{Type I Error}
=
\text{Reject a true }H_0
$$

The significance level $\alpha$ controls the long-run Type I error rate under the test assumptions.

For:

$$
\alpha=0.05
$$

the procedure allows a 5% long-run false-rejection rate when the null hypothesis is true.

## 14. Type II Error

A **Type II error** occurs when we fail to reject $H_0$ even though the alternative is true.

$$
\text{Type II Error}
=
\text{Fail to reject a false }H_0
$$

Its probability is often denoted:

$$
\beta
$$

The probability of correctly rejecting a false null hypothesis is called **statistical power**:

$$
\text{Power}=1-\beta
$$

Power depends on factors such as:

- effect size;
- sample size;
- variability;
- significance level;
- test design.

Formal power analysis is beyond this introductory chapter, but the concept is important.

## 15. Decision Table

| Reality | Fail to Reject $H_0$ | Reject $H_0$ |
|---|---|---|
| $H_0$ true | Correct decision | Type I Error |
| $H_0$ false | Type II Error | Correct decision / Power |

This table is useful because hypothesis testing always involves uncertainty.

No decision rule can eliminate both types of error in every situation.

## 16. Connection to Confidence Intervals

Chapter 11 and hypothesis testing are closely related.

For a two-sided test:

$$
H_0:\mu=\mu_0
$$

at:

$$
\alpha=0.05
$$

there is a close correspondence with a 95% confidence interval.

If the null value $\mu_0$ lies outside the corresponding 95% t-based confidence interval, the two-sided test rejects $H_0$ at the 5% level.

Let's verify this with the current sample.

In [ ]:
critical = t.ppf(0.975, df=dfree)

margin = critical * standard_error
ci_lower = x_bar - margin
ci_upper = x_bar + margin

print(f"95% CI: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"Null value μ0: {mu_0:.2f}")
print(f"Is μ0 inside the CI? {ci_lower <= mu_0 <= ci_upper}")

This relationship is not accidental.

Both confidence intervals and hypothesis tests are built from the same ideas:

- sampling distributions;
- standard errors;
- critical values;
- uncertainty.

Confidence intervals emphasize **estimation**.

Hypothesis tests emphasize **evaluation of a specified claim**.

## 17. Statistical Significance vs Practical Significance

Suppose a very large sample finds that average response time is:

$$
30.1\text{ minutes}
$$

instead of:

$$
30.0\text{ minutes}
$$

With enough data, even a tiny difference can become statistically significant.

But is a difference of 0.1 minutes important for the organization?

Maybe not.

Therefore:

> **Statistical significance does not automatically imply practical importance.**

Always examine the magnitude of the difference, context, and uncertainty—not only the p-value.

## 18. Effect Size in This Example

A simple standardized measure of the difference is:

$$
d=
\frac{\bar{x}-\mu_0}{s}
$$

This expresses the difference in sample-standard-deviation units.

Chapter 12 only introduces this idea; effect sizes can be studied in greater detail later.

In [ ]:
cohens_d = (x_bar - mu_0) / s

print(f"Mean difference: {x_bar - mu_0:.2f} minutes")
print(f"Standardized difference (d): {cohens_d:.3f}")

## 19. Assumptions and Data Quality

A small p-value cannot rescue a bad study design.

Before interpreting a hypothesis test, consider:

- how observations were sampled;
- whether observations are sufficiently independent for the procedure;
- whether important groups were systematically excluded;
- whether extreme outliers affect the result;
- whether the chosen statistical test matches the research question and data;
- whether assumptions or approximations are reasonable.

Statistical inference begins with good data collection and appropriate modeling.

## Important Distinctions

### Null Hypothesis vs Alternative Hypothesis
$H_0$ is the reference claim tested by the procedure. $H_1$ describes the competing possibility.

### p-value vs Probability That $H_0$ Is True
A p-value is calculated assuming $H_0$ is true. It is not $P(H_0\mid\text{data})$.

### Reject vs Prove
Rejecting $H_0$ does not prove $H_1$ with certainty.

### Fail to Reject vs Accept
Failing to reject $H_0$ does not prove or necessarily “accept” $H_0$ as true.

### Statistical vs Practical Significance
A statistically detectable effect may still be too small to matter in practice.

### Confidence Interval vs Hypothesis Test
Both use sampling uncertainty. Confidence intervals emphasize plausible parameter values; hypothesis tests evaluate a specified null claim.

## Exercises

### Exercise 1
Write the null and alternative hypotheses for testing whether a population mean differs from 50.

### Exercise 2
For:

$$
H_0:\mu=30
$$

calculate the t-statistic manually from the dataset using:

$$
t=\frac{\bar{x}-30}{s/\sqrt{n}}
$$

### Exercise 3
Use SciPy to calculate the two-sided p-value and compare it with $\alpha=0.05$.

### Exercise 4
Repeat the decision using:

$$
\alpha=0.01
$$

Does the conclusion change?

### Exercise 5
Explain the meaning of a Type I error in the fictional customer-support scenario.

### Exercise 6
Explain the meaning of a Type II error in the same scenario.

### Exercise 7
Construct the 95% confidence interval and explain its relationship to the two-sided hypothesis test.

### Exercise 8
Explain why a p-value of 0.03 does **not** mean:

> “There is a 3% probability that the null hypothesis is true.”

### Exercise 9
Give an example where a result could be statistically significant but practically unimportant.

## Summary

Hypothesis testing provides a framework for evaluating claims about population parameters using sample data.

The basic workflow is:

$$
\text{Research Question}
\rightarrow
H_0/H_1
\rightarrow
\text{Test Statistic}
\rightarrow
p\text{-value}
\rightarrow
\text{Decision}
\rightarrow
\text{Interpretation}
$$

You learned:

- null and alternative hypotheses;
- one-sided and two-sided tests;
- test statistics;
- null distributions;
- p-values;
- significance level $\alpha$;
- reject vs fail to reject;
- Type I and Type II errors;
- statistical power;
- the relationship between confidence intervals and tests;
- statistical vs practical significance;
- common interpretation mistakes.

The key idea is:

> **Hypothesis testing asks whether the observed evidence would be sufficiently unusual under a specified null model.**

Next:

# 13 t-test

Chapter 12 introduced the general logic. Chapter 13 will apply that framework systematically to common t-tests.